# MRI Reconstruction from Subsampled k-space: Results & Analysis

Authors: Michal Yechezkel (ID: 322556267), Almog Talker (ID: 322546680)

This notebook produces the figures and tables for the report. It is intentionally
**thin**: it loads `results/runs.csv` and `results/samples.csv` and calls the helpers in
`src/analysis.py`. There is no model definition or training here.

**How to run**

- Run from the project root (the `MRI/` folder).
- Produce the runs first, in this order (see the README runbook for details):
  1. `python -m src.run_experiments --sweep configs/experiments/baseline_tuning.yaml` — calibrate the classical baseline on the validation split.
  2. `python -m src.run_experiments --sweep configs/experiments/comparison.yaml` and `--sweep configs/experiments/comparison_baseline_tv.yaml` — the headline comparison.
  3. Optionally the appendix sweeps: `depth_sweep`, `loss_ablation`, `structure_ablation`, `unet_reference`, and `python -m src.eval_crossmask`.
- Then run all cells here.

All statistics are computed **across the test set** (the spread the brief asks for), from
the per-slice metrics in `samples.csv`, pooling the three seeds.

**Contents**
1. Headline results table: PSNR & SSIM (real / imag) per sampling ratio, baseline vs our model
2. PSNR and SSIM vs sampling ratio (line plots with std bands)
3. Sample-wise baseline-vs-model scatter plots with Pearson r
4. Qualitative reconstructions: four categories (both good / both poor / baseline wins / model wins)
5. Appendix: unrolling-depth, loss, and architectural ablations
6. Split verification: age distribution across splits
7. Method and data figures: masks, pipelines, EDA, reconstruction per unrolled stage
8. Appendix: U-Net reference and generalization to unseen undersampling masks

In [ ]:
import os
import sys

sys.path.insert(0, ".")  # run this notebook from the MRI/ project root

import matplotlib.pyplot as plt

from src.config import load_config
import src.analysis as A

# The two models compared in the report.
BASELINE = "classical_cs_tv"       # classical CS: multi-level wavelet L1 + TV, POCS
NAIVE_BASELINE = "classical_cs"    # the single-level variant, kept for the prior ablation
MODEL = "admmnet_softthresh"       # our model: unrolled ADMM-Net

# load_config resolves the right results path for Colab vs local automatically
cfg = load_config("configs/default.yaml")
RESULTS_DIR = cfg["paths"]["results_root"]

FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

df = A.load_results(RESULTS_DIR)          # dataset-wide mean/std per run
samples = A.load_samples(RESULTS_DIR)     # per-sample metrics (for scatter / Pearson r)
print(f"Loaded {len(df)} result rows and {len(samples)} per-sample rows from {RESULTS_DIR}")
print("experiments present:", sorted(df["name"].dropna().unique()))
df.head()

## 1. Headline results table (per sampling ratio)

PSNR and SSIM on the real and imaginary components, one row per sampling ratio for both
the baseline and our model, as `mean +/- std` **across the held-out test set**.

`src/make_report_tables.py` renders this and every other table straight into
`report/tables.md`, so the report never quotes a number typed in by hand.

In [ ]:
table = A.results_table_by_ratio(samples, methods=("zero_filled", NAIVE_BASELINE,
                                                  BASELINE, MODEL), split="test")
table.to_csv(os.path.join(FIG_DIR, "results_by_ratio.csv"))
table

## 2. PSNR and SSIM vs sampling ratio

Metric (average of real and imaginary components) against the sampling ratio, one line
for the baseline and one for our model. The shaded band is one standard deviation across
the test set.

In [ ]:
for base in ("psnr", "ssim"):
    fig = A.plot_metric_vs_ratio(samples, base=base, methods=(BASELINE, MODEL), split="test")
    fig.savefig(os.path.join(FIG_DIR, f"{base}_vs_ratio.png"), dpi=200, bbox_inches="tight")
    plt.show()

## 3. Sample-wise baseline-vs-model scatter (Pearson r)

Each point is one test slice: baseline metric on the x-axis, our model on the y-axis,
coloured by sampling ratio. Points above the `y = x` line are slices where our model
wins. The Pearson correlation coefficient is printed on each plot.

In [ ]:
for base in ("psnr", "ssim"):
    fig = A.scatter_baseline_vs_model(samples, base=base, baseline=BASELINE, model=MODEL,
                                      split="test", seed=0)
    fig.savefig(os.path.join(FIG_DIR, f"scatter_{base}.png"), dpi=200, bbox_inches="tight")
    plt.show()

## 4. Qualitative reconstructions (four required categories)

For a chosen trained ADMM-Net checkpoint, we show four test slices as magnitude images
(zero-filled input / baseline / our model / ground truth): one where both models do
well, one where both do poorly, one where the baseline wins, and one where our model
wins.

In [ ]:
checkpoints = A.list_checkpoints(RESULTS_DIR)
print("Available checkpoints (run_id):", *checkpoints, sep="\n  ")

# Pick a trained ADMM-Net checkpoint (e.g. your best comparison run at ratio 0.3).
CKPT = next((p for rid, p in checkpoints.items() if MODEL in rid), None)
if CKPT is None:
    raise SystemExit("no ADMM-Net checkpoint found; run the comparison sweep first")

fig = A.qualitative_examples(CKPT, baseline=BASELINE, base="psnr")
fig.savefig(os.path.join(FIG_DIR, "qualitative_examples.png"), dpi=150, bbox_inches="tight")
plt.show()

## 5. Appendix: research-environment ablations

Depth (number of unrolled ADMM stages), training loss (image / structural / k-space
consistency), and the architectural variation (soft-threshold vs piecewise-linear
prior, +/- weight sharing). Each is a small, targeted sweep run on complex data at the
0.3 sampling ratio. Missing experiments are skipped with a warning.

In [ ]:
# Depth sweep
fig = A.plot_depth_vs_metric(df, model_name=MODEL, split="test", base="psnr")
fig.savefig(os.path.join(FIG_DIR, "depth_vs_psnr.png"), dpi=200, bbox_inches="tight")
plt.show()

# Loss ablation
fig = A.plot_categorical_ablation(df, experiment="loss_ablation", label_cols=("loss",),
                                  base="psnr", split="test")
fig.savefig(os.path.join(FIG_DIR, "loss_ablation.png"), dpi=200, bbox_inches="tight")
plt.show()

# Structural ablation (nonlinearity x weight sharing)
fig = A.plot_categorical_ablation(df, experiment="structure_ablation",
                                  label_cols=("method", "share_weights"),
                                  base="psnr", split="test")
fig.savefig(os.path.join(FIG_DIR, "structure_ablation.png"), dpi=200, bbox_inches="tight")
plt.show()

## 6. Split verification: age distribution across splits

The professor's update asked us to build our own split and keep the age distribution
approximately the same across it. Each run records the split it used in `split.json`;
here we read one and show the per-split age statistics to confirm they match.

In [ ]:
import glob
import json
import pandas as pd

split_files = sorted(glob.glob(os.path.join(RESULTS_DIR, "*", "split.json")))
if not split_files:
    print("no split.json found yet; run a training/comparison first")
else:
    info = json.load(open(split_files[0]))
    age_table = pd.DataFrame(info["age_stats"]).T
    print(f"Split from: {os.path.dirname(split_files[0])}")
    display(age_table)

## 7. Method and data figures

The figures that describe the *setup* rather than the results: the variable-density
undersampling masks, block diagrams of the two pipelines, the exploratory data analysis
(example slices, age distribution, fully sampled vs undersampled k-space), and the
reconstruction as it evolves through the unrolled ADMM stages.

`masks`, `pipelines` and `results` need no dataset; `eda` needs the dataset and
`per_stage` needs a trained checkpoint. Anything unavailable is skipped with a message.

In [ ]:
from IPython.display import Image, display

import src.figures as F

written = F.main(["--config", "configs/default.yaml"])
for path in written:
    display(Image(filename=path, width=900))

## 8. Appendix: is the advantage structural, and does it survive a new mask?

Two checks on *why* the model wins:

- **Against a plain U-Net** (467,554 parameters against ADMM-Net's 317,320) that maps the
  zero-filled image directly to a clean one, with no data-consistency step. If the
  unrolled model wins while being smaller, the advantage comes from the model-based
  structure rather than from capacity.
- **Under undersampling masks it never saw in training.** Each network trains against one
  mask realization, so we re-evaluate the trained checkpoints on masks drawn with other
  seeds. The classical baseline is included as a control: with no learned parameters, its
  variation shows only how hard each realization intrinsically is.

Requires `configs/experiments/unet_reference.yaml` and `python -m src.eval_crossmask`.

In [ ]:
if "unet" in set(df["method"].unique()):
    fig = A.plot_model_vs_unet(df, model_name=MODEL)
    fig.savefig(os.path.join(FIG_DIR, "model_vs_unet.png"), dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("no U-Net runs yet: "
          "python -m src.run_experiments --sweep configs/experiments/unet_reference.yaml")

try:
    fig = A.plot_crossmask(A.load_crossmask(RESULTS_DIR))
    fig.savefig(os.path.join(FIG_DIR, "crossmask.png"), dpi=200, bbox_inches="tight")
    plt.show()
except FileNotFoundError as exc:
    print(exc)

## 9. Regenerate every table in the report

Writes `report/tables.md` from `results/runs.csv` and `results/samples.csv`, so the report
and the logs can never disagree. Re-run this after adding any experiment.

In [ ]:
from src.make_report_tables import main as make_tables

_ = make_tables(["--config", "configs/default.yaml"])